In [0]:
from pyspark.sql import DataFrame
from delta.tables import DeltaTable 
from pyspark.sql.functions import col, split, trim, lower, when, explode

In [0]:
events = spark.table("dbw_routemind_euskadi_dev.silver.events_cleaned")
location = spark.table("dbw_routemind_euskadi_dev.bronze.municipality_codes")


In [0]:
location_exploded = location.withColumn("name_alt", explode(split(col("municipalityName"), "/")))


In [0]:
# creation of key to join with municipality_codes based on municipality name
location_norm = location_exploded.withColumn(
    "join_key_mun",
    trim(lower(col("name_alt")))
)

In [0]:
events_norm = events.withColumn(
    "join_key_mun_event",
    when(col("municipality") == "Donostia / San Sebastián","san sebastián").otherwise(trim(lower(col("municipality"))))
)

In [0]:
# inner join to ignore events outside from Basque Country
events_location = events_norm.join(
    location_norm,
    events_norm.join_key_mun_event == location_norm.join_key_mun,
    "left"
).select(
    events_norm["id"], 
    events_norm["type_id"],
    events_norm["type_name"], 
    events_norm["location"], 
    events_norm["municipality"], 
    location_norm["municipalityCode"], # municipality code from municipality_codes table
    location_norm["countyId"], # county code from municipality_codes table
    events_norm["startDate"], 
    events_norm["endDate"], 
    events_norm["opening_hour"]
)

In [0]:
events_location.count()

In [0]:
# keep only event from municipalities in Basque Country
events_location_clean = events_location.filter(col("municipalityCode").isNotNull())

In [0]:
events_location_clean.count()

In [0]:
# apply user categorization to filter in WebApp
events_user_category = events_location_clean.withColumn(
    "user_category",
    when(
        (col("type_name") == "Concierto") | (col("type_name") == "Festival"),
        "conciertos-festivales"
    ).when(
        (col("type_name") == "Teatro") | (col("type_name") == "Exposición"),
        "teatro-arte").otherwise("eventos-culturales")
)

display(events_user_category.head(20))

In [0]:
target_table = "dbw_routemind_euskadi_dev.gold.events_user_category"
delta_path = "abfss://gold@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/events/data"

if not spark.catalog.tableExists(target_table):
    print(f"Table {target_table} doesn't exist. Creating table...")
    
    events_user_category.write \
        .format("delta") \
        .option("path", delta_path) \
        .saveAsTable(target_table)
        
    print(f"table {target_table} created. rows processed: {events_user_category.count()}")

else:
    delta_target = DeltaTable.forName(spark, target_table)
    (
        delta_target.alias("t")
        .merge(
            events_user_category.alias("s"),
            "t.id = s.id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE completed on {target_table}. rows processed: {events_user_category.count()}")